In [ ]:
# Define the parameter to hold the CSV file path
%load_ext autoreload
%autoreload 2
import itertools
import logging
from logging import getLogger

import pandas as pd
from nemo_platform import NeMoPlatform
from nemo_platform.beta.safe_synthesizer.job_builder import SafeSynthesizerJobBuilder

logger = getLogger(__name__)
logging.getLogger("httpcore").setLevel(logging.DEBUG)


# leave commented out - we inject this path for testing in CI
# csv_filepath = "womens-clothing-ecommerce.csv"
data = pd.read_csv(csv_filepath).head(250)  # noqa

client = NeMoPlatform(
    base_url="http://localhost:8080",
    inference_base_url="http://localhost:8080",
)
builder: SafeSynthesizerJobBuilder = SafeSynthesizerJobBuilder(client).with_data_source(data).synthesize()

In [ ]:
unsloths, quant = [True, False], [True, False]

for unsloth, quant in itertools.product(unsloths, quant):
    builder = builder.with_train(use_unsloth=unsloth, quantize_model=quant)
    print(f"Training with unsloth: {unsloth}, quantize_model: {quant}")
    job = builder.create_job()
    job.wait_for_completion()
    current_status = job.fetch_status()
    expected_status = "completed"

    assert current_status == expected_status, (
        f"Job status mismatch. Expected: '{expected_status}', Got: '{current_status}'"
    )
    print(f"Job completed with unsloth: {unsloth}, quantize_model: {quant}")
    current_shape = job.fetch_data().shape
    expected_shape = (1000, 11)

    assert expected_shape == current_shape, (
        f"Result shape mismatch. Expected: '{expected_shape}', Got: '{current_shape}'"
    )